# Imports

In [66]:
from openai import OpenAI
import pandas as pd
import glob
import os
from dotenv import load_dotenv
import json
import re
import time
import csv

pd.set_option('display.max_colwidth', None)

load_dotenv()


True

# OpenAI

In [2]:
# Activate OpenAI Client

client = OpenAI()

In [3]:
# Create a vector store
vector_store = client.vector_stores.create(name="DRI_Examples")

# Ready the files for upload to OpenAI
file_paths = ["data/05_LLM_data/inputs/dri_example_statements.txt"]
file_streams = [open(path, "rb") for path in file_paths]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

vector_store_id = vector_store.id

print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [27]:
name = f"DRI Considerations Assistant"

instructions = f"""
You are an expert political Scientist. Your task is to find
Consideration Statements for a DRI Survey on health costs in Switzerland from
different media outlets grouped by political leaning on several subtopics.
The following is a short introduction to the concept of DRI:

"
The Deliberative Reason Index (DRI) is a tool used to measure the quality and
improvement of group reasoning during deliberative processes. It focuses on
evaluating reasoning at the group level, rather than individual level, emphasizing
how group members collectively construct an understanding of an issue and agree
on the terms of reasonableness.

Consideration statements are specific statements in a survey designed to capture
participants' opinions or evaluations of relevant factors, arguments, or aspects
of an issue under discussion. These statements reflect the various interests,
values, beliefs, or facts that participants consider important when reasoning
about the issue.

In the context of the Deliberative Reason Index (DRI), consideration statements
help identify the range of viewpoints or concerns that individuals bring to a
deliberative process. They serve as a way to assess how participants weigh
different aspects of an issue before arriving at preferences for actions or policies.
"

Consideration statements are typically concisely formulated and express a single
consideration relevant to the topic at hand. Considerations express feelings, opinions, 
reasons and arguments. 
Good consideration statements are formulated in easy language in a clear, personal and polarizing manner.
They should not include demands for policy actions or express preferences for
specific outcomes.

Good examples include:
- "Enforcing health prevention is paternalistic."
- "People that don't take care of their health should pay more for their insurance."
- "Health prevention is too expansive."
- "Health prevention can help to lower overall health costs."
- "It's the responsibility of the state to ensure good overall health of its population."

Bad examples include:
- "Introducing a unified health insurance system could reduce costs by decreasing expenditures on marketing, administrative fees, and high management salaries currently spread across multiple insurance companies."
- "There is a significant lack of financial incentives for health prevention in Switzerland, leading to spiraling healthcare costs."
- "Closing underutilized hospitals in Switzerland would result in significant cost savings and improve overall efficiency in health service delivery."

Additionally read through the dri_example_statements.csv and familiarize yourself with the
semantic structure of consideration statements.
Be prepared to produce a list of relevant Consideration statements in JSON Format, based
on the data that will be provided to you. 
Always make sure to equally represent supportive and critical formulations.
When prompted, make sure to produce all statements in English and in close semantic
similarity to the examples. Do not include anything but a list of statements in
JSON Format in your responses
"""


assistant = client.beta.assistants.create(
    name=name,
    instructions=instructions,
    model="gpt-4o",
    tools=[{"type": "file_search"}],
)

asssistant_id = assistant.id

assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store_id]}},
)


In [28]:
# Final prompt creation with the selected prompt similarity
prompt = f"""
    Create a list of 25 consideration statements for and against the topic of health prevention in
    Switzerland from the data provided below.   
    Make sure to do include statements from the diverse range of
    perspectives and opinions represented in the data. 
    Don't be nice and don't try to bridge gaps.
    For this scientific method we need polarizing, strong statements to assess people's agreement.
    Include critical and supportive considerations equally.
    Consideration statements are typically concisely formulated in simple language in a relatable and personal way. 
    They express a single premise for an argument or viewpoint. 
    A consideration could be a reason, a feeling, an argument or an opinion but NOT a policy option.

    Format your response as a list of consideration statements in JSON format.
    Make sure not to include anything else but the JSON object in your response.
"""

In [ ]:
# # Final prompt creation with the selected prompt similarity
# prompt = f"""
#     Create a list of 25 consideration statements for and against the topic of health prevention in
#     Switzerland from the data provided below. 
#     When creating statements try to take the perspective of the 
    
#     Make sure to do include statements from the diverse range of
#     perspectives and opinions represented in the data. Don't be nice and don't try to bridge gaps.
#     For this scientific method we need polarizing, strong statements to assess people's agreement.
#     Include critical and supportive considerations equally.
#     Consideration statements are typically concisely formulated in a relatable and personal way. They express a single
#     premise for an argument or viewpoint. A consideration could be a reason, a feeling, an argument or an opinion.

#     Format your response as a list of consideration statements in JSON format.
#     Make sure not to include anything else but the JSON object in your response.
# """

In [ ]:
# # Final prompt creation with the selected prompt similarity
# prompt = f"""
#     Create a list of 25 consideration statements for and against the topic of health prevention in
#     Switzerland from the data provided below. Make sure to do include statements from the diverse range of
#     perspectives and opinions represented in the data. The statements should be
#     in close semantic similarity to the examples provided in the dri_example_statements.csv
#     file but the content should be representative of the perspectives described in the data.
#     Make sure to capture the whole range of viewpoints and concerns for and against health 
#     prevention measures that is represented in the data.
#     Consideration statements are typically concisely formulated in a relatable and personal way. They express a single
#     premise for an argument or viewpoint. A consideration could be a reason, a feeling, an argument or an opinion.

#     Format your response as a list of consideration statements in JSON format.
#     Make sure not to include anything else but the JSON object in your response.
# """

In [29]:
# ========== User Configurable Parameters ===========
input_dir = "data/97_side_data/health_prevention_dri_survey_7.25/Transcripts"
output_file = "merged_transcripts.txt"
prefixes = ["S3", "S4"]
# ==================================================

# Build glob patterns and collect matching files
patterns = [os.path.join(input_dir, f"{prefix}_*.txt") for prefix in prefixes]
matched_files = []
for pattern in patterns:
    matched_files.extend(glob.glob(pattern))
matched_files.sort()

print(f"Found {len(matched_files)} files with prefixes: {prefixes}")

# Optionally, display the matched filenames
for f in matched_files:
    print(f"  - {os.path.basename(f)}")


if not matched_files:
    print(f"No files to merge in `{input_dir}` with prefixes {prefixes}.")
else:
    print(f"Merging {len(matched_files)} files into `{output_file}`...")
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for filepath in matched_files:
            filename = os.path.basename(filepath)
            outfile.write(f"\n===== {filename} =====\n")
            with open(filepath, 'r', encoding='utf-8') as infile:
                outfile.write(infile.read())
    print("Merge complete!")


Found 52 files with prefixes: ['S3', 'S4']
  - S3_CH_Swiss_Breakout_aggr.txt
  - S3_DE_NWS_Breakout_Flavia.txt
  - S3_DE_NWS_Breakout_Flurina.txt
  - S3_DE_NWS_Breakout_Osi.txt
  - S3_DE_NWS_Breakout_aggr.txt
  - S3_DE_NWS_Plenum_1.txt
  - S3_DE_NWS_Plenum_2.txt
  - S3_DE_OST_Breakout_Jennifer.txt
  - S3_DE_OST_Breakout_Lukas.txt
  - S3_DE_OST_Breakout_Noemi.txt
  - S3_DE_OST_Breakout_aggr.txt
  - S3_DE_OST_Plenum_1.txt
  - S3_DE_OST_Plenum_2.txt
  - S3_DE_ZS_Breakout_Amadea.txt
  - S3_DE_ZS_Breakout_Anais.txt
  - S3_DE_ZS_Breakout_Nora.txt
  - S3_DE_ZS_Breakout_aggr.txt
  - S3_DE_ZS_Plenum_1.txt
  - S3_DE_ZS_Plenum_2.txt
  - S3_FR_Romandie_Breakout_Elsa.txt
  - S3_FR_Romandie_Breakout_Micael.txt
  - S3_FR_Romandie_Breakout_Sarah.txt
  - S3_FR_Romandie_Breakout_aggr.txt
  - S3_FR_Romandie_Plenum_1.txt
  - S3_FR_Romandie_Plenum_2.txt
  - S3_IT_Ticino_Breakout_Amelie.txt
  - S3_IT_Ticino_Breakout_Andrea.txt
  - S3_IT_Ticino_Breakout_aggr.txt
  - S3_IT_Ticino_Plenum_1.txt
  - S3_IT_Ticino

In [37]:
abschlussbericht_file_path = f'data/97_side_data/health_prevention_dri_survey_7.25/DE_Abschlussbereicht_Bev-Rat_E11_digital.pdf'

abschlussbericht_file = client.files.create(
file=open(abschlussbericht_file_path, "rb"),
purpose="assistants"
)

transcript_file_path = f'data/97_side_data/health_prevention_dri_survey_7.25/Transcripts/merged_transcripts.txt'

transcript_file = client.files.create(
file=open(transcript_file_path, "rb"),
purpose="assistants"
)


# # Create a thread and attach the file to the message
thread = client.beta.threads.create(
messages=[
    {
    "role": "user",
    "content": prompt,
    # Attach the new file to the message.
    "attachments": [
        { "file_id": abschlussbericht_file.id, "tools": [{"type": "file_search"}] },
        { "file_id": transcript_file.id, "tools": [{"type": "file_search"}] }
    ],
    }
]
)

# Create a thread and attach the file to the message
thread = client.beta.threads.create(
messages=[
    {
    "role": "user",
    "content": prompt,
    # Attach the new file to the message.
    "attachments": [
        { "file_id": transcript_file.id, "tools": [{"type": "file_search"}] }
    ],
    }
]
)

/tmp/ipykernel_25707/2425645732.py:17: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create(
/tmp/ipykernel_25707/2425645732.py:32: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create(


In [38]:
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id, assistant_id=assistant.id
)

/tmp/ipykernel_25707/3012490424.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create_and_poll(


In [39]:
messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))
messages

/tmp/ipykernel_25707/1182473377.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))


[Message(id='msg_FzUcm09G1uluOY3nyOF9IOAP', assistant_id='asst_gTwcSykuOfUK5LyrAX0y2Qu6', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='```json\n[\n    "Health prevention invades personal freedom too much.",\n    "People should take more responsibility for their own health without relying on prevention programs.",\n    "Investing in prevention saves healthcare costs in the long run.",\n    "Mandatory health check-ups feel intrusive and unnecessary to me.",\n    "Prevention programs can help improve quality of life and increase longevity.",\n    "Healthy living feels burdensome with constant prevention advice.",\n    "Prevention education should start early in schools to be most effective.",\n    "Health prevention approaches often seem to overlook cultural diversity.",\n    "Promoting a healthy lifestyle empowers individuals.",\n    "Government should not dictate how individuals manage their health.",\n    "Preventive care prevents diseas

In [40]:
statements_df = pd.DataFrame()

message_content = messages[0].content[0].text.value

# Remove the markdown ```json``` block
#  and newlines using regex
cleaned_content = re.sub(r'```json|```', '', message_content).strip()

# Load the cleaned content as a JSON array
statements = json.loads(cleaned_content)

# Create the column name dynamically
column_name = f"Consideration Statements"

# Pad the list of statements with NaN to match the length of the DataFrame
padded_statements = statements + [None] * (len(statements_df) - len(statements))

# Add the new column to the DataFrame
statements_df[column_name] = padded_statements


In [41]:
statements_df

,Consideration Statements
0,Health prevention invades personal freedom too much.
1,People should take more responsibility for their own health without relying on prevention programs.
2,Investing in prevention saves healthcare costs in the long run.
3,Mandatory health check-ups feel intrusive and unnecessary to me.
4,Prevention programs can help improve quality of life and increase longevity.
5,Healthy living feels burdensome with constant prevention advice.
6,Prevention education should start early in schools to be most effective.
7,Health prevention approaches often seem to overlook cultural diversity.
8,Promoting a healthy lifestyle empowers individuals.
9,Government should not dictate how individuals manage their health.


In [42]:
statements_df.to_csv("data/97_side_data/health_prevention_dri_survey_7.25/health_prevention_dri_statements.csv")

# GROK

In [65]:
import os
from xai_sdk import Client
from xai_sdk.chat import system, user
import PyPDF2

# Initialize the Grok (XAI) client
client = Client(
    api_key=os.getenv("XAI_API_KEY"),
    timeout=3600,  # allow longer reasoning times
)

# 1. Read in your example‑statements file with latin‑1
examples_path = "data/05_LLM_data/inputs/dri_example_statements.txt"
with open(examples_path, "r", encoding="latin-1") as f:
    example_statements = f.read()

# 1. Read in the transcript file
transcript_path = "data/97_side_data/health_prevention_dri_survey_7.25/Transcripts/merged_transcripts.txt"
with open(transcript_path, "r", encoding="utf-8") as f:
    transcript = f.read()

# 3. Extract text from the PDF
pdf_path = "data/97_side_data/health_prevention_dri_survey_7.25/DE_Abschlussbereicht_Bev-Rat_E11_digital.pdf"
pdf_text = []
with open(pdf_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pdf_text.append(text)
pdf_content = "\n\n".join(pdf_text)

# 3. Define the system instructions and user prompt
instructions = """
You are an expert political Scientist. Your task is to find
Consideration Statements for a DRI Survey on health costs in Switzerland from
different media outlets grouped by political leaning on several subtopics.
The following is a short introduction to the concept of DRI:

"
The Deliberative Reason Index (DRI) is a tool used to measure the quality and
improvement of group reasoning during deliberative processes. It focuses on
evaluating reasoning at the group level, rather than individual level, emphasizing
how group members collectively construct an understanding of an issue and agree
on the terms of reasonableness.

Consideration statements are specific statements in a survey designed to capture
participants' opinions or evaluations of relevant factors, arguments, or aspects
of an issue under discussion. These statements reflect the various interests,
values, beliefs, or facts that participants consider important when reasoning
about the issue.

In the context of the Deliberative Reason Index (DRI), consideration statements
help identify the range of viewpoints or concerns that individuals bring to a
deliberative process. They serve as a way to assess how participants weigh
different aspects of an issue before arriving at preferences for actions or policies.
"

Consideration statements are typically concisely formulated and express a single
consideration relevant to the topic at hand. Considerations express feelings, opinions, 
reasons and arguments. 
Good consideration statements are formulated in easy language in a clear, personal and polarizing manner.
They should not include demands for policy actions or express preferences for
specific outcomes.

Good examples include:
- "Enforcing health prevention is paternalistic."
- "People that don't take care of their health should pay more for their insurance."
- "Health prevention is too expansive."
- "Health prevention can help to lower overall health costs."
- "It's the responsibility of the state to ensure good overall health of its population."

Bad examples include:
- "Introducing a unified health insurance system could reduce costs by decreasing expenditures on marketing, administrative fees, and high management salaries currently spread across multiple insurance companies."
- "There is a significant lack of financial incentives for health prevention in Switzerland, leading to spiraling healthcare costs."
- "Closing underutilized hospitals in Switzerland would result in significant cost savings and improve overall efficiency in health service delivery."

Additionally read through the dri_example_statements.csv and familiarize yourself with the
semantic structure of consideration statements.
Be prepared to produce a list of relevant Consideration statements in JSON Format, based
on the data that will be provided to you. 
Always make sure to equally represent supportive and critical formulations.
When prompted, make sure to produce all statements in English and in close semantic
similarity to the examples. Do not include anything but a list of statements in
JSON Format in your responses
"""

# These are examples for DRI statements in different contexts:
# """ + example_statements

prompt = """
Create a list of 25 consideration statements for and against the topic of health prevention in
Switzerland from the data provided below.   
Make sure to include statements from the diverse range of
perspectives and opinions represented in the data. 
Don't be nice and don't try to bridge gaps.
For this scientific method we need polarizing, strong statements to assess people's agreement.
Include critical and supportive considerations equally.
Consideration statements are typically concisely formulated in simple language in a relatable and personal way. 
They express a single premise for an argument or viewpoint. 
A consideration could be a reason, a feeling, an argument or an opinion but NOT a policy option.

Format your response as a list of consideration statements in JSON format.
Make sure not to include anything else but the JSON object in your response.

Context:
""" + pdf_content

# 4. Create and run the chat
chat = client.chat.create(model="grok-3")
chat.append(system(instructions))
chat.append(user(prompt))

response = chat.sample()
print(response.content)

{
  "consideration_statements": [
    "I believe health prevention is crucial to keep people healthy and reduce long-term costs.",
    "I think focusing on health prevention feels like the state is meddling too much in my personal choices.",
    "I feel that health prevention programs are a waste of money when we could treat illnesses instead.",
    "I’m convinced that early health prevention can save lives by catching problems before they get worse.",
    "I find health prevention initiatives annoying because they often target behaviors I enjoy.",
    "I strongly support health prevention as it helps create a healthier society for everyone.",
    "I’m skeptical about health prevention because I don’t think it actually changes people’s habits.",
    "I believe investing in health prevention is smarter than spending so much on curing diseases.",
    "I feel health prevention is unfair because it puts pressure on individuals instead of fixing the system.",
    "I think health prevention 

In [67]:
raw = response.content

# 1. (Optional) strip outer quotes if you accidentally captured the repr()
if raw.startswith("'") and raw.endswith("'"):
    raw = raw[1:-1]

# 2. Parse JSON
data = json.loads(raw)
statements = data["consideration_statements"]

# 3. Write to CSV
with open("consideration_statements.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["statement"])      # header
    for s in statements:
        writer.writerow([s])

print(f"Saved {len(statements)} statements to consideration_statements.csv")

Saved 25 statements to consideration_statements.csv


# _____________________OLD_____________________

# Load embedded data with similarity scores

In [3]:
df = pd.read_csv('data/05_LLM_data/full_merged_data.csv')
# rename main similarity score to similarity_score_0
df['similarity_score_0'] = df['similarity_score']
df.drop('similarity_score', axis=1, inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: 'data/05_LLM_data/full_merged_data.csv'

# Set lists for medium according to political alignment

In [ ]:
rechts = ['WEWO', 'WEW', 'WDAY', 'NZZO']
rechtsliberal = ['NZZS', 'NZZ', 'SAW']
mitte = ['LUZ', 'ZWAS', 'ZWAO', 'ZWA', 'ZWSO', 'ZWAF', 'SRFA', 'SRF', 'SRFV', 'AZO', 'AZM', 'RTS', 'RTSV']
linksliberal = ['BAZ', 'NNBS', 'TLM', 'TLMD', 'HEU', 'NNHEU', 'TDG', 'NNTDG', 'BZ', 'NNBE', 'BLI', 'BLIO', 'NNTA', 'TA', 'TAZT', 'NNTA', 'TPS', 'TPSO', 'TAS', 'TLMD', 'BLI', 'BLIO']
links = ['WOZ', 'SBLI']

medium_groups = {
    'rechts': rechts,
    'rechtsliberal': rechtsliberal,
    'mitte': mitte,
    'linksliberal': linksliberal,
    'links': links
}
similarity_columns = [
    'similarity_score_0', 'similarity_score_1', 'similarity_score_2',
    'similarity_score_3', 'similarity_score_4', 'similarity_score_5',
    'similarity_score_6', 'similarity_score_7'
]


# Get Top scoring 'text' element per similarity_score

In [ ]:
# Iterate over each group in the dictionary
for group_name, group_list in medium_groups.items():
    # Assuming df_selected is your DataFrame, and you have a 'text' column for each row

    # Create an empty DataFrame to store the top 500 results for each column
    top_500_all_scores = pd.DataFrame()

    # Filter the DataFrame for the current group
    df_selected = df[df['medium_code'].isin(group_list)]

    # Dictionary to store top 500 rows for each similarity score column
    top_500_dict = {}

    # Iterate over each similarity score column
    for col in similarity_columns:
        # Sort by the similarity score column in descending order and select top 500 rows
        top_500 = df_selected.sort_values(by=col, ascending=False).head(500)

        # Add the top 500 rows to the dictionary with the column name as the key
        top_500_dict[col] = top_500['text'].reset_index(drop=True)

    # Convert the dictionary to a DataFrame
    top_500_all_scores = pd.DataFrame(top_500_dict)

    # Save the top 500 elements for each column to a CSV file, using the group name
    top_500_all_scores.to_csv(f'data/06_LLM_data/inputs/{group_name}_top_500_similarity_scores.csv', index=False)


# Extract Consideration Statements with GPT for each similarity score

In [4]:
# Activate OpenAI Client

client = OpenAI()

## Setting up the Assistant
### (We only need to do this once)

In [7]:
# Create a vector store
vector_store = client.vector_stores.create(name="DRI_Examples")

# Ready the files for upload to OpenAI
file_paths = ["data/05_LLM_data/inputs/dri_example_statements.txt"]
file_streams = [open(path, "rb") for path in file_paths]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

vector_store_id = vector_store.id

print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [9]:
name = f"DRI Considerations Assistant"

instructions = f"""
You are an expert political Scientist. Your task is to find
Consideration Statements for a DRI Survey on health costs in Switzerland from
different media outlets grouped by political leaning on several subtopics.
The following is a short introduction to the concept of DRI:

"
The Deliberative Reason Index (DRI) is a tool used to measure the quality and
improvement of group reasoning during deliberative processes. It focuses on
evaluating reasoning at the group level, rather than individual level, emphasizing
how group members collectively construct an understanding of an issue and agree
on the terms of reasonableness.

Consideration statements are specific statements in a survey designed to capture
participants' opinions or evaluations of relevant factors, arguments, or aspects
of an issue under discussion. These statements reflect the various interests,
values, beliefs, or facts that participants consider important when reasoning
about the issue.

In the context of the Deliberative Reason Index (DRI), consideration statements
help identify the range of viewpoints or concerns that individuals bring to a
deliberative process. They serve as a way to assess how participants weigh
different aspects of an issue before arriving at preferences for actions or policies.
"

Consideration statements are typically concisely formulated and express a single
premise for an argument or viewpoint. They are designed to capture the essence of
a particular consideration that participants may have when discussing a given topic.
Good consideration statements are clear, specific, and relevant to the issue at hand.
They should not include demands for policy actions or express preferences for
specific outcomes.

Good examples include:
- "Private solutions are more efficient than government intervention in managing healthcare costs."
- "A growing older population is leading to higher utilization of healthcare services, increasing overall costs."
- "There is a significant lack of financial incentives for health prevention in Switzerland, leading to spiraling healthcare costs."ArithmeticError

Bad examples include:
- "Introducing a unified health insurance system could reduce costs by decreasing expenditures on marketing, administrative fees, and high management salaries currently spread across multiple insurance companies."
- "We need to introduce cost targets in the health system to manage the rapid growth of healthcare costs."
- "Closing underutilized hospitals in Switzerland would result in significant cost savings and improve overall efficiency in health service delivery."

Additionally read through the dri_example_statements.csv and familiarize yourself with the
semantic structure of consideration statements.
Be prepared to produce a list of the top 5 Consideration statements in JSON Format, based
on the media outlets that will be procided to you.
When prompted, make sure to produce all statements in English and in close semantic
similarity to the examples. Do not include anything but a list of statements in
JSON Format in your responses
"""


assistant = client.beta.assistants.create(
    name=name,
    instructions=instructions,
    model="gpt-4o",
    tools=[{"type": "file_search"}],
)

asssistant_id = assistant.id

assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store_id]}},
)


## Prompt the Assistant
### (Do this every run)

In [ ]:
def customize_prompt(political_leaning, similarity_score):
    # Define a dictionary mapping similarity scores to corresponding prompts
    similarity_prompts = {
        0: """
        Focus specifically on the topic of health costs in Switzerland.
        """,
        1: """
        Focus specifically on the topic of the health system of Switzerland and non-incremental policy reforms.
        """,
        2: """
        Focus specifically on the topic of Increasing utilisation of healthcare services per person.
        """,
        3: """
        Focus specifically on the topic of hospital planning and financing.
        """,
        4: """
        Focus specifically on the topic of the design/configuration/organisation of compulsory health insurance.
        """,
        5: """
        Focus specifically on the topic of a lack of incentives for health prevention.
        """,
        6: """
        Focus specifically on the topic of the coordination of health services.
        """,
        7: """
        Focus specifically on the topic of the financial burden of health costs on households (costs distribution).
        """
    }

    # Retrieve the correct prompt based on the similarity score
    prompt_similarity = similarity_prompts.get(similarity_score, similarity_prompts[0])

    # Final prompt creation with the selected prompt similarity
    prompt = f"""
        Create a list of 5 consideration statements for a DRI Survey on health costs in
        Switzerland from the media outlets provided below. All media outlets provided come from
        {political_leaning}-leaning media outlets. Make sure to do justice to the
        perspectives and opinions described in the media outlets. The statements should be
        in close semantic similarity to the examples provided in the dri_example_statements.csv
        file but the content should be representative of the political opinion of the media outlets.
        Make sure to capture the whole range of viewpoints and concerns that is represented in
        the media outlets. Then boil it down to the top 5 most important consideration statements.
        Consideration statements are typically concisely formulated and express a single
        premise for an argument or viewpoint. They are designed to capture the essence of
        a particular consideration that participants may have when discussing a given topic.

        {prompt_similarity}

        Format your response as a list of consideration statements in JSON format.
        Make sure not to include anything else but the JSON object in your response.
    """

    return prompt


In [ ]:
def get_consideration_statements(political_leaning, similarity_score_list):
    statements_df = pd.DataFrame(index=range(10))

    for similarity_score in similarity_score_list:
        # Path to the directory containing your CSV files
        csv_path = f'data/06_LLM_data/input_data/{political_leaning}_top_500_similarity_scores.csv'
        df = pd.read_csv(csv_path)
        print("df loaded with shape:")
        print(df.shape)

        txt_file_path = f'data/06_LLM_data/input_data/{political_leaning}_top_500_similarity_score_{similarity_score}.txt'

        # Save the column as a text file, each value on a new line
        df[f'similarity_score_{similarity_score}'].to_csv(txt_file_path, index=False, header=False)
        print(f"File {txt_file_path} has been created with content:")
        print(df[f'similarity_score_{similarity_score}'].head())

        message_file = client.files.create(
        file=open(txt_file_path, "rb"),
        purpose="assistants"
        )
        print("message_file:")
        print(message_file)

        # Create a thread and attach the file to the message
        thread = client.beta.threads.create(
        messages=[
            {
            "role": "user",
            "content": customize_prompt(political_leaning, similarity_score),
            # Attach the new file to the message.
            "attachments": [
                { "file_id": message_file.id, "tools": [{"type": "file_search"}] }
            ],
            }
        ]
        )

        if os.path.exists(txt_file_path):
            os.remove(txt_file_path)
            print(f"File {txt_file_path} has been removed.")
        else:
            print(f"File {txt_file_path} does not exist.")

        print("Waiting for the assistant to generate the consideration statements...")
        run = client.beta.threads.runs.create_and_poll(
            thread_id=thread.id, assistant_id=assistant.id
        )

        messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))
        print("messages:")
        print(messages)

        message_content = messages[0].content[0].text.value

        # Remove the markdown ```json``` block and newlines using regex
        cleaned_content = re.sub(r'```json|```', '', message_content).strip()

        # Load the cleaned content as a JSON array
        statements = json.loads(cleaned_content)

        # Create the column name dynamically
        column_name = f"Statements, similarity_score_{similarity_score}"

        # Pad the list of statements with NaN to match the length of the DataFrame
        padded_statements = statements + [None] * (len(statements_df) - len(statements))

        # Add the new column to the DataFrame
        statements_df[column_name] = padded_statements

        # Wait for 20 seconds before continuing to the next iteration to avoid rate limiting
        print(f"Waiting for 20 seconds before the next iteration...")
        time.sleep(20)


    return statements_df

In [ ]:
# Run the Generation of DRI Statements

# Define the political leaning and similarity score
political_leaning_list = ['right', 'right-liberal', 'centrist', 'left-liberal', 'left']
political_leaning= political_leaning_list[0]

similarity_score_list = [i for i in range(8)]

# Define vector store and assistant IDs if not already defined
vector_store_id = ''
asssistant_id = ''

df = get_consideration_statements(political_leaning, similarity_score_list)


In [ ]:
# Save

df.to_csv('data/06_LLM_data/outputs/consideration_statements/centrist_consideration_statements.csv', index=False)

# Extract Policy Preferences with GPT for each similarity score

In [ ]:
# Activate OpenAI Client

client = OpenAI()


## Setting up the Assistant
### (We only need to do this once)

In [ ]:
# Create a vector store
vector_store = client.beta.vector_stores.create(name="DRI_policy_options")

# Ready the files for upload to OpenAI
file_paths = ["data/06_LLM_data/similarity_scores/dri_example_policy_options.txt"]
file_streams = [open(path, "rb") for path in file_paths]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

vector_store_id = vector_store.id

In [ ]:
name = f"DRI Policy Assistant"

instructions = f"""
You are an expert political Scientist. Your task is to find
Policy Options for a DRI Survey on health costs in Switzerland from
different media outlets grouped by political leaning on several subtopics.
The following is a short introduction to the concept of DRI:

"
The Deliberative Reason Index (DRI) is a tool used to measure the quality and
improvement of group reasoning during deliberative processes. It focuses on
evaluating reasoning at the group level, rather than individual level, emphasizing
how group members collectively construct an understanding of an issue and agree
on the terms of reasonableness.

Policy Options are specific possible actions or strategies that will be ranked
by preferance within the DRI survey. These options reflect the various policy changes
that are discussed in the public discourse on to the issue at hand.

In the context of the Deliberative Reason Index (DRI), policy options
help identify the range of preferences and opinions concerning policy demands
that individuals bring to a deliberative process.
"

Policy options are typically concisely formulated and express a single
strategy or action. They are designed to capture the main solutions discussed
within the public media discourse on a given topic.
Good policy options are clear, specific, and relevant to the issue at hand.
They should include demands for policy actions and express preferences for
specific outcomes. They can include a statement of the problem if necessary.

Additionally read through the dri_example_policy_options.csv and familiarize yourself with the
semantic structure of consideration statements.
Be prepared to produce a list of the top policy options in JSON Format, based
on the media outlets that will be procided to you.
When prompted, make sure to produce all statements in English and in close semantic
similarity to the examples. Do not include anything but a list of statements in
JSON Format in your responses
"""


assistant = client.beta.assistants.create(
    name=name,
    instructions=instructions,
    model="gpt-4o",
    tools=[{"type": "file_search"}],
)

asssistant_id = assistant.id

assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store_id]}},
)

## Prompt the Assistant
### (Do this every run)

In [15]:
def customize_prompt(political_leaning):
    # Define a dictionary mapping similarity scores to corresponding prompts

    # Final prompt creation with the selected prompt similarity
    prompt = f"""
        Create a list of policy options for a DRI Survey on health costs in
        Switzerland from the media outlets provided below. All media outlets provided come from
        {political_leaning}-leaning media outlets. Make sure to do justice to the
        perspectives and opinions described in the media outlets. The statements should be
        in close semantic similarity to the examples provided in the dri_example_policy_options.csv
        file but the content should be representative of the political opinion of the media outlets.
        Make sure to capture the whole range of policy proposals that is represented in
        the media outlets. Then boil it down to the top 5 most important policy options.
        Policy options are typically concisely formulated and express a single
        action or strategy to a give issue.

        Format your response as a list of consideration statements in JSON format.
        Make sure not to include anything else but the JSON object in your response.
    """

    return prompt


In [ ]:
def get_consideration_statements(political_leaning):
    policies_df = pd.DataFrame(index=range(10))

    # Path to the directory containing your CSV files
    csv_path = f'data/06_LLM_data/input_data/{political_leaning}_top_500_similarity_scores.csv'
    df = pd.read_csv(csv_path)
    print("df loaded with shape:")
    print(df.shape)

    txt_file_path = f'data/06_LLM_data/input_data/{political_leaning}_top_500_similarity_score.txt'

    # Save the column as a text file, each value on a new line
    df.to_csv(txt_file_path, index=False, header=False)
    print(f"File {txt_file_path} has been created")

    message_file = client.files.create(
    file=open(txt_file_path, "rb"),
    purpose="assistants"
    )
    print("message_file:")
    print(message_file)

    # Create a thread and attach the file to the message
    thread = client.beta.threads.create(
    messages=[
        {
        "role": "user",
        "content": customize_prompt(political_leaning),
        # Attach the new file to the message.
        "attachments": [
            { "file_id": message_file.id, "tools": [{"type": "file_search"}] }
        ],
        }
    ]
    )

    if os.path.exists(txt_file_path):
        # os.remove(txt_file_path)
        print(f"File {txt_file_path} has been removed.")
    else:
        print(f"File {txt_file_path} does not exist.")

    print("Waiting for the assistant to generate the policy options...")
    run = client.beta.threads.runs.create_and_poll(
        thread_id=thread.id, assistant_id=assistant.id
    )

    messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))
    print("messages:")
    print(messages)

    message_content = messages[0].content[0].text.value

    # Remove the markdown ```json``` block and newlines using regex
    cleaned_content = re.sub(r'```json|```', '', message_content).strip()

    # Now you can load the cleaned content as a JSON array
    statements = json.loads(cleaned_content)

    # Create the column
    column_name = f"Statements, similarity_score"

    # Pad the list of statements with NaN to match the length of the DataFrame
    padded_statements = statements + [None] * (len(policies_df) - len(statements))

    # Add the new column to the DataFrame
    policies_df[column_name] = padded_statements


    return policies_df

In [ ]:
# Run the Generation of DRI Statements

# Define the political leaning and similarity score
political_leaning_list = ['right', 'right-liberal', 'centrist', 'left-liberal', 'left']
political_leaning= political_leaning_list[0]

similarity_score_list = [i for i in range(8)]

# Define vector store and assistant IDs if not already defined
vector_store_id = ''
asssistant_id = ''

df = get_consideration_statements(political_leaning_list[4])

# Rename accordingly
df_left = df
# df_leftliberal = df
# df_centrist = df
# df_rightliberal = df
# df_right = df

In [37]:
# Merged DFs
merged_df = pd.concat([df_right, df_rightliberal, df_centrist,
                       df_leftliberal, df_left], axis=1)

# Clean DF
df_cleaned = merged_df.dropna()


# Saving

In [ ]:
# Save DF
df_cleaned.to_csv('data/06_LLM_data/outputs/policy_options/policy_preferences.csv', index=False)